(course-core-native-forms)=

# Module 4: Native Forms

**Welcome back, Apprentice Master.** In Module 1 and Module 2, you discovered that MolSysMT treats every external file format or third-party object as a **Form**, while querying and modifying its internal data through **Attributes**.

While MolSysMT is form-agnostic, it is not homeless. When you want maximum computational speed, zero-copy structural queries, and complete control over your data, you rely on **Native Forms**.

At the center of this ecosystem is **`molsysmt.MolSys`**, a modular orchestrator that unifies domain-specific native objects: **`Topology`** (stable atom inventory and hierarchy), **`ChemicalStates`** (covalent bonds and state-dependent chemistry), **`Structures`** (coordinates, boxes and trajectories), and **`MolecularMechanics`** (force field parameters, charges and energies). It can also carry named `Interactions` analyses aligned to its atom and structure indices.

```{admonition} Glossary: Native Form
:class: dropdown info
A **Native Form** is any data structure built directly within MolSysMT (`molsysmt.MolSys`, `molsysmt.Topology`, `molsysmt.Structures`, `molsysmt.TopologyDict`, `file:h5msm`) designed for optimal memory layout and maximum analytical performance. See the {ref}`user-foundations` guide for a full list of native forms.
```

(course-core-native-forms-learning-outcomes)=
```{admonition} Learning Outcomes
:class: dropdown learning-outcomes

By the end of this module, you will be able to:
- Instantiate and work with the **`molsysmt.MolSys`** native orchestrator object.
- Understand the modular architecture of component objects (`Topology`, `ChemicalStates`, `Structures`, `MolecularMechanics`).
- Leverage disk-based **H5MSM** containers for high-performance storage.
- Access lightweight dictionary forms (`TopologyDict`, `StructuresDict`, `MolSysDict`) for zero-overhead Python data manipulation.
```

### 1. The Native Orchestrator: `molsysmt.MolSys`

Let's begin by importing MolSysMT, loading a molecular system, and converting it to MolSysMT's primary in-memory native object: **`molsysmt.MolSys`**.

In [1]:
import molsysmt as msm
from molsysmt import systems

# Load T4 Lysozyme and convert it into the native MolSys object
lysozyme_file = systems['T4 lysozyme L99A']['181l.bcif.gz']
molsys = msm.convert(lysozyme_file, to_form='molsysmt.MolSys')

When inspecting `molsys`, notice the difference between Python's internal class implementation (`type()`) and MolSysMT's canonical form string (`msm.get_form()`):

In [2]:
print(f"Python class type : {type(molsys)}")
print(f"MolSysMT form name: {msm.get_form(molsys)}")

Python class type : <class 'molsysmt.native.molsys.MolSys'>
MolSysMT form name: molsysmt.MolSys


:::{hint}
:class: dropdown
**molsysmt.MolSys**: The primary in-memory native container of MolSysMT. It orchestrates modular component objects (`topology`, `chemical_states`, `structures`, and `molecular_mechanics`). See API doc: {func}`molsysmt.native.MolSys`.
:::

### 2. Modular Domain Components: `Topology`, `ChemicalStates`, `Structures`, and `MolecularMechanics` 

Rather than storing all data in a single monolithic array, `molsysmt.MolSys` delegates domain responsibilities to specialized native component objects:
- **`molsys.topology` (`molsysmt.Topology`)**: Manages the stable atom inventory, groups/residues, chains, molecules, and entities. Its `bonds` property remains a convenient view of the reference chemical state. *(Explored in depth in [Module 05: Combined Forms](../Common_Core/index.md))*
- **`molsys.chemical_states` (`molsysmt.ChemicalStates`)**: Owns the state-dependent covalent bonds, component membership, and chemical assignments. `Topology` does not expose a `chemical_states` property; `Topology.bonds` remains a compatibility view of the reference state. Inspect one state's bonds with `molsys.chemical_states.get_bonds(chemical_state=0)`.
- **`molsys.structures` (`molsysmt.Structures`)**: Manages spatial coordinates array `(n_structures, n_atoms, 3)`, periodic box vectors `(n_structures, 3, 3)`, time steps, velocities, and frame observables. *(Explored in depth in [Module 21: Data Analyst](../Common_Core/index.md) & [Module 40: The Physics Lab](../Common_Core/index.md))*
- **`molsys.molecular_mechanics` (`molsysmt.MolecularMechanics`)**: Provides a minimal, experimental native container for force-field settings, charges, non-bonded parameters, and energy information.

These domains remain useful independently. For example, a standalone `molsysmt.ChemicalStates(n_atoms=3)` starts with no chemical states, while a native `molsysmt.Topology` keeps one reference state for compatibility. Converting a topology to `molsysmt.MolSys` produces zero structures; MolSysMT does not invent coordinates. Named analyses in `molsys.interactions` are remapped by `copy()`, `extract()`, and `remove()`; their local-to-source index maps retain positional provenance. Each analysis declares which atoms and structures were evaluated. Adding atoms keeps the previous atom search scope fixed, and appended structures stay unevaluated. When coordinates change, `Interactions.invalidate_structures()` marks affected structures unevaluated without removing them from the system. H5MSM 0.5 stores these named analyses; H5MSM 0.4 cannot. The Buch hydrogen-bond and disulfide candidate detectors can optionally return a complete `molsysmt.Interactions` analysis for explicit attachment. Buch retains rectangular default arrays for equal counts and uses aligned lists with empty entries for varying counts; the disulfide default remains pair and distance lists. See [Querying interaction results](../../user/tools/interactions/result.md) for scopes and experimental limits. Attaching an analysis declares that its local atom and structure indices match the target system. You are responsible for that correspondence when loading an independent file; source labels and equal axis sizes do not authenticate molecular origin. Each analysis also retains producer versions in `software`; saving or loading with another version does not replace them. An empty dictionary means unknown producer versions. Luzard-Chandler also offers the sparse output, with D-A distances in nm, H-D-A angles in rad, and donor-anchored D-H/D-A periodic images. Its default arrays become aligned lists when per-structure counts vary.

An experimental partial `MolSys` can hold any combination of topology, chemical states, and structures. `msm.get` and `msm.has_attribute` use the domains actually present: absent chemistry is unavailable rather than an empty bond collection. `msm.convert(..., to_form='file:h5msm')` writes the modular H5MSM 0.5 schema.

You can compose a topology-free MolSys from separate chemical and structural
domains using `msm.convert([chemical_states, structures])`; the chemistry can
also be a ChemicalStatesDict. Supplying the pair declares matching atom indices.
Ordered atom and structure index selections remap both domains together. A
reference state is retained, while several states remain unassigned to individual
structures until you declare that association. See {ref}`Tutorial_Convert` and
{ref}`Cookbook_Saving_pi_pi_interactions` for executable examples.

Named `Interactions` analyses use integer atom and structure indices. Their public constructors and queries validate arguments with ArgDigest, including invalidated and recalculated results. Fractional indices and booleans are refused before encoding. Leave `skip_digestion=False` for ordinary use; `True` is only for inputs already checked against the complete contract. See {ref}`Querying interaction results <user-tools-interactions-result>`.

Let's inspect the form of each component attribute inside our `molsys` instance:

In [3]:
print(f"Topology component form: {msm.get_form(molsys.topology)}")
print(f"Structures component form: {msm.get_form(molsys.structures)}")
print(f"Molecular Mechanics component form: {msm.get_form(molsys.molecular_mechanics)}")

Topology component form: molsysmt.Topology
Structures component form: molsysmt.Structures
Molecular Mechanics component form: molsysmt.MolecularMechanics


:::{hint}
:class: dropdown
**molsysmt.Topology**: Native object representing the stable atom inventory and hierarchy. Covalent bonds belong to `ChemicalStates` and remain accessible through `Topology.bonds` for compatibility. See API docs for native classes: {func}`molsysmt.native.Topology`, {class}`molsysmt.native.ChemicalStates`, {func}`molsysmt.native.Structures`, and {func}`molsysmt.native.MolecularMechanics`.
:::

When converting a GRO file to `mdtraj.Topology` or `mdtraj.Trajectory`, MolSysMT
reuses the native candidate-bond policy on the complete first structure before
selection. MDTraj still reads the coordinate series, times and boxes. An inferred
graph does not establish complete chemistry; inspect the
{ref}`GRO conversion policy <user-foundations-native-world-file-handlers-molsysmt-grofilehandler>`
before treating it as a preparation topology.


### 3. High-Performance Disk Storage: H5MSM Files (`file:h5msm`)

The **H5MSM** format (`file:h5msm`) is the disk-based counterpart of `molsysmt.MolSys`. Built on top of HDF5, it stores system topology, trajectory coordinates, box vectors, and physical observables in binary form with maximum I/O performance.

Unlike third-party formats, **H5MSM** allows chunked execution and partial trajectory loading without materializing the full dataset into memory. *(Trajectory I/O workflows are covered in [Module 14: The Virtual Lab](../Common_Core/index.md) & [Module 47: Pipeline Developer](../Common_Core/index.md))*.
Public `msm.convert` and the explicit `msm.h5msm` API write and read the modular 0.5 schema, where `Topology`, `ChemicalStates`, `Structures`, and interaction analyses are optional sibling layers. A file without topology can load as a partial `MolSys`; named interaction analyses can supply atom and structure index domains for `extract()` even when structures are absent. Topology, chemical states and several named analyses also round-trip without Structures, preserving evaluated-empty coverage and original producer versions. The writer records shared axis associations; neither coordinates nor per-structure chemical-state assignments are inferred. Biological assemblies retain their chain indices, rotations, and unit-bearing translations. Sparse alternate atom locations preserve their site coordinates, occupancies, B factors, and IDs; structure IDs retain string or integer type. H5MSM 0.5 rejects nonempty molecular-mechanics data; persistence of that experimental domain is deferred to H5MSM 0.6 after MolSysMT 1.0. Reading a legacy 0.3 or 0.4 file emits a warning pointing to `msm.h5msm.migrate_to_05`, which writes a separate translated file. The legacy file-form adapters remain for older layouts; see the [H5MSM 0.5 guide](../../user/tools/form/file/h5msm_05.md) for the versioned API.

In [4]:
# Inspect an H5MSM file from the demo repository
h5msm_file = systems['T4 lysozyme L99A']['181l.h5msm']
print(f"Form of the file: {msm.get_form(h5msm_file)}")

Form of the file: file:h5msm


:::{hint}
:class: dropdown
**file:h5msm**: Native disk-based HDF5 container for MolSysMT systems, supporting complete trajectory datasets and structural states. See User Guide: {ref}`user-foundations`.
:::

### 4. Lightweight Native Dictionaries: `TopologyDict`, `StructuresDict`, and `MolSysDict` 

For high-speed Python pipelines, API serialization, or low-latency inspection, MolSysMT provides dictionary-based native forms:
- **`molsysmt.TopologyDict`**: Lightweight dictionary containing pure topological arrays and connectivity maps.
- **`molsysmt.StructuresDict`**: Lightweight dictionary containing raw coordinate arrays and frame observables.
- **`molsysmt.MolSysDict`**: Dictionary representation bundling topology, structures, and mechanics data.
- **`molsysmt.ChemicalStatesDict`** and **`molsysmt.InteractionsDict`**: Versioned, typed columns for complete chemical-state and sparse-interaction results. Convert them with `msm.convert()`; their NumPy arrays preserve types and compact storage but are not directly JSON-compatible.

Because these forms use pure Python dictionaries, all internal data is transparently accessible via standard key lookup. Let's convert `molsys.topology` into a `TopologyDict` and explore its data:

In [5]:
# Convert topology component to a native TopologyDict
topo_dict = msm.convert(molsys.topology, to_form='molsysmt.TopologyDict')

print(f"Form found: {msm.get_form(topo_dict)}")
print(f"Dictionary keys: {list(topo_dict.data.keys())}")
print(f"\nNumber of atom records: {len(topo_dict.data['atoms'])}")
print(f"Number of group records: {len(topo_dict.data['groups'])}")

Form found: molsysmt.TopologyDict
Dictionary keys: ['format', 'kind', 'version', 'metadata', 'atoms', 'groups', 'bonds', 'chains', 'molecules', 'entities']

Number of atom records: 1441
Number of group records: 302


:::{hint}
:class: dropdown
**molsysmt.TopologyDict**: Lightweight native Python dictionary form holding topology attributes for zero-overhead iteration and serialization. See API doc: {func}`molsysmt.native.TopologyDict`.
:::

--- 

### 🏆 Challenge 4: The Native Architect

1. Load the **SARS-CoV-2 Protease** using its PDB ID: `'pdb_id:6LU7'`.
2. Convert it into a native **`molsysmt.MolSys`** object.
3. Use `msm.get_form()` on its `topology` and `structures` attributes to verify their native forms.
4. Convert the native object into a **`molsysmt.TopologyDict`** and inspect its `'atoms'` key length.

Mastering native forms gives you maximum performance and flexibility. In [Module 5: Combined Forms](../Common_Core/05_Combined_Forms.ipynb), we will learn how to compose systems using multiple complementary forms.

```{key-takeaway}
MolSysMT native objects (`MolSys`, `Topology`, `Structures`, `H5MSM`, and `Dict` forms) provide a modular, high-performance architecture for structural biology.
```

(course-core-native-forms-see-also)=
:::{seealso}
:class: dropdown
**API Documentation for Native Forms in this Module:**
- {func}`molsysmt.native.MolSys` — Primary native in-memory orchestrator container.
- {func}`molsysmt.native.Topology` — Stable atom inventory and hierarchy.
- {class}`molsysmt.native.ChemicalStates` — Covalent bonds and state-dependent chemistry.
- {func}`molsysmt.native.Structures` — Native structural coordinate and trajectory object.
- {func}`molsysmt.native.MolecularMechanics` — Native molecular mechanics and forcefield object.
- {func}`molsysmt.native.TopologyDict` — Lightweight topology dictionary form.
- {func}`molsysmt.native.StructuresDict` — Lightweight structural dictionary form.
- {func}`molsysmt.native.MolSysDict` — Lightweight complete system dictionary form.

**Related Course Modules & Guides:**
- Previous Module: [Module 3: Molecular Elements](../Common_Core/03_Molecular_Elements.ipynb)
- Next Module: [Module 5: Combined Forms](../Common_Core/05_Combined_Forms.ipynb)
- User Guide: {ref}`user-foundations`
:::